In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "src")))
import numpy as np, sympy as sp
from IPython.display import Image, display, Markdown
sp.init_printing(use_unicode=True)
def show(path):
    p = path[0] if isinstance(path, tuple) else path
    display(Image(filename=p))


# 14. Full reproducibility study

Runs the complete verification suite (symbolic identities, analytic benchmarks, numerical checks), prints the summary and shows how each check is recorded.
This takes a few minutes because the symbolic checks are exact computations.

In [2]:
from desitter_perturbations.validation import validation_report as vr
recs, meta = vr.run_all(fast=False, symbolic=True)
print(meta)
print(len(recs), "records;", sum(r.passed for r in recs), "passed;", sum(r.kind == "negative_control" for r in recs), "negative controls")

{'python': '3.13.16', 'numpy': '2.5.3', 'scipy': '1.18.1', 'sympy': '1.14.0', 'platform': 'Linux-6.18.44-fc-v114-x86_64-with-glibc2.39', 'seconds': 122.0, 'timings': {'symbolic': 111.4, 'analytic': 0.0, 'numerical': 10.5}}
128 records; 128 passed; 9 negative controls


In [3]:
import collections
by = collections.defaultdict(list)
for r in recs: by[r.category].append(r)
for c, rs in by.items(): print(f"{c:22s} {len(rs):3d} records   all passed: {all(r.passed for r in rs)}")

background              18 records   all passed: True
metric_expansion         7 records   all passed: True
linearized_einstein      5 records   all passed: True
bianchi                  4 records   all passed: True
gauge                    6 records   all passed: True
svt                     10 records   all passed: True
sectors                  7 records   all passed: True
constraints              2 records   all passed: True
action                   9 records   all passed: True
canonical                7 records   all passed: True
analytic                22 records   all passed: True
numerical               31 records   all passed: True


In [4]:
worst = sorted([r for r in recs if r.kind in ("numerical", "benchmark")], key=lambda r: -r.residual / max(r.tolerance, 1e-300))[:8]
print("checks closest to their tolerance (residual / tolerance):")
for r in worst: print(f"  {r.id:36s} {r.residual:.2e} / {r.tolerance:.1e}")

checks closest to their tolerance (residual / tolerance):
  num.no_horizon_spike                 1.10e+00 / 2.0e+00
  bench.subhorizon.x10000              5.00e-09 / 1.0e-08
  bench.subhorizon.x1000               5.00e-07 / 1.0e-06
  bench.subhorizon.x100                5.00e-05 / 1.0e-04
  num.small_k_amplification_E          1.97e-01 / 5.0e-01
  bench.superhorizon.x0.01             1.25e-09 / 5.0e-09
  bench.superhorizon.x0.1              1.24e-05 / 5.0e-05
  bench.superhorizon.x0.001            1.25e-13 / 5.1e-13


What the report establishes and what it does not: it shows that the implemented equations are mutually consistent, reproduce independent analytic results and are numerically converged;
it does not show that the free-field de Sitter model is a complete description of gravity (see `docs/limitations.md`).